# Can we classify breast cancer subtype with fewer genes than what is in the PAM50 assay?

For this analysis, we are going to see if we can accurately classify the breast cancer subtype of patients using fewer genes than the 50 used in the PAM50 assay. We will be using PCA on proteomic data to achieve this. This data is originally from a 2016 article by Merkins et.al. titled **Proteogenomics connects somatic mutations to signalling in breast cancer**. The data is publicly available on Kaggle: https://www.kaggle.com/datasets/piotrgrabo/breastcancerproteomes.

# Reading in the Data

The data for this analysis will come from three files:
1) clinical_data_breast_cancer.csv: contains patient IDs and breast cancer subtype of each patient
2) 77_cancer_proteomes_CPTAC_itraq.csv: contains protein expressions of each patient
3) PAM50_proteins.csv: contains the list of the PAM50 genes

Since the data is coming from three different files, we will have to merge them at some point (see Data Manipulation section).

In [1]:
# Import pandas
import pandas as pd

# Load in clinical data containing cancer subtypes of study participants
clinical_data = pd.read_csv("clinical_data_breast_cancer.csv")

# Only keep relevant columns
cd = clinical_data.loc[:,['Complete TCGA ID','PAM50 mRNA']]
cd.head(5)

,Complete TCGA ID,PAM50 mRNA
0,TCGA-A2-A0T2,Basal-like
1,TCGA-A2-A0CM,Basal-like
2,TCGA-BH-A18V,Basal-like
3,TCGA-BH-A18Q,Basal-like
4,TCGA-BH-A0E0,Basal-like


In [2]:
# Load in proteome data
itraq_vals = pd.read_csv("77_cancer_proteomes_CPTAC_itraq.csv")
# Load in PAM50 genes
pam50 = pd.read_csv("PAM50_proteins.csv")

# Define the PAM50 genes in a list
pam50_list = list(pam50['GeneSymbol'].unique())

# Filter proteome data to only include PAM50 genes
her2_vals = itraq_vals[itraq_vals['gene_symbol'].isin(pam50_list)]
patient_ids = her2_vals.columns[3:].tolist()
columns_of_interest = ['gene_symbol'] + patient_ids
her2_vals = her2_vals[columns_of_interest]
her2_vals.head(5)

,gene_symbol,AO-A12D.01TCGA,C8-A131.01TCGA,AO-A12B.01TCGA,BH-A18Q.02TCGA,C8-A130.02TCGA,C8-A138.03TCGA,E2-A154.03TCGA,C8-A12L.04TCGA,A2-A0EX.04TCGA,...,AO-A12B.34TCGA,A2-A0SW.35TCGA,AO-A0JL.35TCGA,BH-A0BV.35TCGA,A2-A0YM.36TCGA,BH-A0C7.36TCGA,A2-A0SX.36TCGA,263d3f-I.CPTAC,blcdb9-I.CPTAC,c4155b-C.CPTAC
194,CENPF,0.490373,0.764109,-1.108807,-0.179342,0.106156,-0.409316,-2.742664,0.179349,-2.506723,...,-1.651943,0.293752,1.359573,-1.358787,0.000077,0.944990,1.457488,-0.905015,-1.778314,0.236681
290,MKI67,-0.625898,1.743696,-1.507899,-1.463969,1.883845,0.739026,-3.307172,-0.692468,-2.953194,...,-3.573312,0.039676,-1.177327,-3.278318,-1.320640,-0.918145,1.385987,-3.401099,-1.692017,-2.300132
320,KRT5,-3.056549,0.658864,-4.121399,2.182497,-2.639094,0.943825,-5.903908,1.607704,-0.803378,...,-5.760191,-5.945578,-6.194613,-2.701464,0.291277,-1.125656,5.625404,2.897253,-0.323888,-0.287098
831,KRT14,-2.629851,1.172945,-4.528806,2.399947,-1.579696,1.806910,-4.650701,2.523387,-0.098028,...,-4.632492,-1.899662,-2.263673,0.235850,-0.781806,-0.474116,2.166540,2.405068,-0.614352,-0.009573
1463,ERBB2,9.668177,0.197406,-1.491270,-5.187379,-2.960521,-0.504402,-3.894260,0.039529,-1.643795,...,-4.404098,-4.308558,-1.770736,-3.623104,-3.322351,-2.779049,-4.891209,-1.951586,-0.817467,-0.578304


# Data Manipulation

In order to join the datasets, they need to have a common column which link the data together. These are usually some form of ID column, such as the Complete TCGA ID column in the clinical data. The itRAQ values data also has these IDs, but as the column names. We can pivot the data by adjusting the indices and transposing the data.

In [3]:
#Pivoting the data so that the patient IDs can be their own column (important for merging later on)

pivoted_her2_vals = her2_vals.set_index('gene_symbol')
pivoted_her2_vals = pivoted_her2_vals.transpose()
pivoted_her2_vals = pivoted_her2_vals.reset_index()
pivoted_her2_vals = pivoted_her2_vals.rename(columns={'gene_symbol':'ind','index':'Patient ID'})

pivoted_her2_vals.head(5)

gene_symbol,Patient ID,CENPF,MKI67,KRT5,KRT14,ERBB2,EGFR,MAPT,MAPT,ANLN,...,CXXC5,UBE2T,CDC20,MELK,SLC39A6,BCL2,EXO1,BIRC5,CCNE1,MYC
0,AO-A12D.01TCGA,0.490373,-0.625898,-3.056549,-2.629851,9.668177,0.402748,-2.938445,-3.094647,2.631480,...,NaN,-0.187771,-0.465886,0.840875,-1.677402,-3.544203,1.347578,-1.989805,0.151301,NaN
1,C8-A131.01TCGA,0.764109,1.743696,0.658864,1.172945,0.197406,1.245807,-5.522250,-5.461532,2.419693,...,NaN,-0.053563,2.298256,0.901737,-1.919636,-2.312280,2.395405,0.865306,0.108353,NaN
2,AO-A12B.01TCGA,-1.108807,-1.507899,-4.121399,-4.528806,-1.491270,-2.386456,2.017416,2.050673,-1.992907,...,NaN,-0.518483,-1.064463,0.213186,-0.360509,-1.643701,0.069070,-2.256197,-0.396538,NaN
3,BH-A18Q.02TCGA,-0.179342,-1.463969,2.182497,2.399947,-5.187379,-1.922286,-8.218295,-8.188186,-1.233137,...,-2.624816,-0.717949,0.539915,0.001309,-0.641005,-0.627623,NaN,-0.888563,-3.581595,NaN
4,C8-A130.02TCGA,0.106156,1.883845,-2.639094,-1.579696,-2.960521,-2.422622,1.149155,1.158994,0.506300,...,-2.350465,1.923204,-0.080796,-0.592456,-0.835166,-2.540698,NaN,-0.005359,1.132755,NaN


Now that the TCGA IDs have become their own column, we need to reformat them to how they are formatted in the clinical data so that that they can be matched when merging.

In [4]:
#Reformatting TCGA ID values in our pivoted data to match the format in the clinical data

def reformat_patient_id(patient_id_str: str):
    """Reformats Patient ID to match Complete TCGA ID."""
    patient_id_prefix = patient_id_str.split('.')[0] # split string into list, access element at index 0
    tcga_id = "TCGA-" + patient_id_prefix # string concatenation
    return tcga_id

pivoted_her2_vals['Complete TCGA ID'] = \
    pivoted_her2_vals.apply(lambda x: reformat_patient_id(x['Patient ID']),
                                                    axis=1)

Now we can merge the two datasets and drop any unecessary columns for easier readability.

In [5]:
#Merging the dataset and dropping the extra Patient ID column
pam50_df = pd.merge(cd, pivoted_her2_vals, on='Complete TCGA ID')
pam50_df.drop(columns=['Patient ID'], inplace=True)
pam50_df.head()

,Complete TCGA ID,PAM50 mRNA,CENPF,MKI67,KRT5,KRT14,ERBB2,EGFR,MAPT,MAPT,...,CXXC5,UBE2T,CDC20,MELK,SLC39A6,BCL2,EXO1,BIRC5,CCNE1,MYC
0,TCGA-A2-A0CM,Basal-like,2.160157,2.623021,4.768355,4.933663,-4.419112,-0.271711,-6.013418,-6.318320,...,-1.292949,3.541400,3.177722,NaN,-1.792547,NaN,NaN,2.516489,2.556897,NaN
1,TCGA-BH-A18Q,Basal-like,-0.179342,-1.463969,2.182497,2.399947,-5.187379,-1.922286,-8.218295,-8.188186,...,-2.624816,-0.717949,0.539915,0.001309,-0.641005,-0.627623,NaN,-0.888563,-3.581595,NaN
2,TCGA-A7-A0CE,Basal-like,1.761673,-0.583448,5.542903,2.080457,-3.099008,-1.009547,-3.556670,-3.556670,...,-3.329417,1.922643,2.935811,NaN,-2.253124,-3.430418,NaN,0.997851,1.209322,NaN
3,TCGA-D8-A142,Basal-like,-0.415696,-1.232627,-2.823113,0.668726,-3.130366,-0.462687,-6.195666,-6.427009,...,-2.320664,-1.232627,-1.438667,NaN,NaN,-0.910915,NaN,-0.075910,-0.101213,-7.659635
4,TCGA-AO-A0J6,Basal-like,-0.075090,-1.159168,0.650452,0.661745,-2.062567,-0.394103,-0.893795,-0.913557,...,-0.577605,-2.003281,-4.278717,-0.125906,-3.451542,-0.176722,1.567966,-0.859917,NaN,NaN


The final step is to impute missing values with the means of each column.

In [6]:
# Impute NaNs with the mean of each column in final_df
pam50_df = pam50_df.fillna(pam50_df.mean(numeric_only=True))

# Write data to its own .csv file
pam50_df.to_csv("pam50_df.csv", index=False)